In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import Callable, Sequence
from scipy import linalg

from popt.config import sect
from popt.config import _1M, _3M, _6M, _1Y
fctr = ["SPY", "AGG", "GLD"]
from popt.alpha.modules.features import (
    momentum, 
    drawdown, 
    volatility, 
    volatility_downside, 
    sharpe_like, 
    skewness
)
from popt.alpha.modules.features import FeatureBuilder, FeatureView

In [2]:
D0 = "1993-01-29"
D1 = "2026-03-24"
rd = pd.read_parquet("../../../data/return/return_d.parquet")#.loc[D0:D1]
rf = pd.read_parquet("../../../data/return/ffr_d.parquet").reindex(rd.index)
rx = rd - rf.values

In [3]:
fb = FeatureBuilder(ret_d=rx, tickers=sect, factors=fctr, lookback=252, first_date=D0, final_date=D1)
# add separate methods for adding constant and non-return features (macro and similar?)
fb.add_feature(name="mom_1m", regress=True, z_scale=True, lookback=_1M, callback=momentum)
fb.add_feature(name="mom_3m", regress=True, z_scale=True, lookback=_3M, callback=momentum)
fb.add_feature(name="mdd_1m", regress=True, z_scale=True, lookback=_1M, callback=drawdown)
fb.add_feature(name="mdd_3m", regress=True, z_scale=True, lookback=_3M, callback=drawdown)
fb.add_feature(name="vol_1m", regress=True, z_scale=True, lookback=_1M, callback=volatility)
fb.add_feature(name="vol_3m", regress=True, z_scale=True, lookback=_3M, callback=volatility)
fb.add_feature(name="vld_1m", regress=True, z_scale=True, lookback=_1M, callback=volatility_downside)
fb.add_feature(name="vld_3m", regress=True, z_scale=True, lookback=_3M, callback=volatility_downside)
fb.add_feature(name="shp_3m", regress=True, z_scale=True, lookback=_3M, callback=sharpe_like)
fb.add_feature(name="skw_3m", regress=True, z_scale=True, lookback=_3M, callback=skewness)
fb.consolidate()

In [4]:
# TODO
# - features is new list of feature names
# - target is name of target feature
# - x is new ndarray holding feature data
# - y is new ndarray holding target data

fv = FeatureView(fb, target="mom_3m", subset=None)
# fv.add_mask(["XLK"], ["mom_1m"], exclude=False) 
# fv.add_mask(["XLV"], ["mom_1m"], exclude=True)  
# fv.apply_masking()
fv.x.shape, fv.y.shape

((8344, 9, 10), (8344, 9))

In [6]:
Xx = fv.x[-252-21:-21,:,:]
yy = fv.y[-252:,:]
Xx.shape, yy.shape

((252, 9, 10), (252, 9))

(8344, 9)

In [8]:
# pair-wise ranking loss

def fit_pooled(
        xt: np.ndarray,
        yt: np.ndarray,
        gamma: float,
    ) -> np.ndarray:
    T, N, F = xt.shape
    TN = T*N
    assert yt.shape == (T, N), yt.shape
    
    xt = xt.reshape(TN, F)
    yt = yt.reshape(TN, 1)

    # X @ B = y
    # X.T @ X @ B = X.T @ y
    # B = (X.T @ X.T).inv @ X.T @ y
    # NOTE dont regularize the constant term!! I_f[0, 0] = 0.0
    theta = np.linalg.solve(xt.T @ xt + gamma * np.eye(F), xt.T @ yt)

    return theta

for _ in range(5000):
    theta = fit_pooled(Xx, yy, 4)
theta

array([[ 0.12199978],
       [-0.03215634],
       [-0.06689568],
       [-0.2007178 ],
       [ 0.2511134 ],
       [-0.33975764],
       [-0.25756088],
       [ 0.58120381],
       [ 0.39969264],
       [ 0.20033534]])

In [ ]:
# NOTE prediction loop
# takes in alpha predictor and calls it iteratively over sliding window of returns
def alpha_predictor(
        X: pd.DataFrame,
        y: pd.DataFrame,
        lap_graph: np.ndarray,  # includes edge weights
        gamma: float = 0.0,
        halflife: int = 126,
        lookback: int = 504,    # 2 yrs lookback for training
        horizon: int = 21,      # 21 days ahead, 1 trading month
        pooled: bool = True,
    ) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    """ Backtest for alpha predictor - Eval different strategies """
    # T train samples, N assets, F features, L iterations
    timeloss = lookback+horizon
    T = lookback
    _, N = y.shape
    F = X.shape[1] // N
    L = y.shape[0] - timeloss

    # Precomputed matricies
    I_n = np.eye(N)
    I_f = np.eye(F)
    # XXX This is contigent on the const feature being the first feature
    I_f[0, 0] = 0.0  # NOTE no regularization for bias term, thus 0.0
    L_nf = np.kron(lap_graph, I_f) # kronecker product [N, N] (x) [F, F] -> [NF, NF]
    G_nf = gamma * np.kron(I_n, I_f)
    G_f = gamma * I_f

    beta: int = (0.5) ** (1.0 / halflife)
    w_root = np.array([beta ** (T-i-1) for i in range(T)])
    w_root = np.sqrt(w_root)
    predictions = np.empty((L, N), dtype=np.float32)
    thetas = np.empty((L, N, F), dtype=np.float32)

    for t in range(timeloss, y.shape[0]):              # y[t-1]: t:t+H-1  <- The prediction we seek
        X_trn = X.iloc[t-timeloss:t-horizon].values    # [T, NF]      r: t-L-H:t-H-1
        y_trn = y.iloc[t-timeloss:t-horizon].values    # [T, N]       r: t-L:t-1
        X_tst = X.iloc[t].values                       # [NF,]        r: t-H+1:t
        y_tst = y.iloc[t].values                       # [N,]         r: t+1:t+H

        mu = X_trn.mean(axis=0)
        sd = X_trn.std(axis=0, ddof=1)
        mu[0::F] = 0.0  # avoid scaling bias term
        sd[0::F] = 1.0
        X_trn = (X_trn - mu) / sd
        X_tst = (X_tst - mu) / sd

        match pooled:
            case True:
                Theta = train_shared(X_trn, y_trn, w_root, G_f, T, N, F)
                y_prd = X_tst.reshape(N, F) @ Theta
            case False:
                Theta = train(X_trn, y_trn, w_root, L_nf, G_nf, T, N, F)  # Training            
                y_prd = (X_tst.reshape(N, F) * Theta).sum(axis=1) # Testing - sum([N, F] * [N, F]) over F
            case _: 
                raise ValueError("invalid pooled argument.")

        assert y_prd.shape == y_tst.shape, (y_prd.shape, y_tst.shape)  # [N,]
        predictions[t-timeloss] = y_prd
        thetas[t-timeloss] = Theta
    
    df_ref = y.iloc[timeloss:]
    df_prd = pd.DataFrame(
        data=predictions,
        columns=df_ref.columns,
        index=df_ref.index
    )

    assert df_ref.shape == df_prd.shape, (df_ref.shape, df_prd.shape)
    return df_prd, df_ref, thetas


In [ ]:
# NOTE Add to FeatureBuilder later on, not needed right now. 
#     def save_to_npz(self, file_path: str, verbose=False) -> None:
#         self.consolidate()
#         np.savez(
#             file=file_path,  # "../../../data/feature/sector.npz"
#             features=self.features,
#             names=self.names,
#             lookbacks=self.lookbacks,
#             timeline=self.timeline,
#             tickers=self.tickers,
#             factors=self.factors,
#         )
#         if verbose == True: print(f"stored {file_path}")

#         T = ret_d.shape[0]
#         N = len(tickers)
#         K = len(factors)
#         lookback_regression = lookback
#         rr = ret_d[tickers].values
#         xx = ret_d[factors].values
#         ee = rolling_regression(self.rr, self.xx, lookback)
#         timeline = ret_d.index                
#         tickers = tickers
#         factors = factors

#         F: int = None
#         features:   np.ndarray = None
#         __features: list[np.ndarray] = []
#         names: list[str] = []
#         lookbacks: list[str] = []
#         callbacks: list[Callable[..., np.ndarray]] = []

In [ ]:
# r, x = rx[sect].values, rx[fctr].values
# r_eps = rolling_target(r, x, lookback=252)

# mom_1m = standard_scale( rolling_feature(r_eps, _1M, momentum) )
# mom_3m = standard_scale( rolling_feature(r_eps, _3M, momentum) )
# mdd_1m = standard_scale( rolling_feature(r_eps, _1M, drawdown) )
# mdd_3m = standard_scale( rolling_feature(r_eps, _3M, drawdown) )
# vol_1m = standard_scale( rolling_feature(r_eps, _1M, volatility) )
# vol_3m = standard_scale( rolling_feature(r_eps, _3M, volatility) )
# vld_1m = standard_scale( rolling_feature(r_eps, _1M, volatility_downside) )
# vld_3m = standard_scale( rolling_feature(r_eps, _3M, volatility_downside) )
# shp_3m = standard_scale( rolling_feature(r_eps, _3M, sharpe_like) )
# skw_3m = standard_scale( rolling_feature(r_eps, _3M, skewness) )

# feature_set = [
#     ("MOM 1M", mom_1m, _1M),
#     ("MOM 3M", mom_3m, _3M),
#     ("MDD 1M", mdd_1m, _1M),
#     ("MDD 3M", mdd_3m, _3M),
#     ("VOL 1M", vol_1m, _1M),
#     ("VOL 3M", vol_3m, _3M),
#     ("VLD 1M", vld_1m, _1M),
#     ("VLD 3M", vld_3m, _3M),
#     ("SHP 3M", shp_3m, _3M),
#     ("SKW 3M", skw_3m, _3M),
# ]

# timeline = rx.index.to_numpy()
# assets = rx.columns.to_numpy()
# names = np.array([e[0] for e in feature_set])
# lookbacks = np.array([e[2] for e in feature_set])
# features = np.r_[*[e[1][None] for e in feature_set]]
# features.shape


In [ ]:
# NOTE potential interactions - stroner / weaker certain days
# Multiply by best features and see if any improvements

timeline = rx.index

day_of_wk = timeline.day_of_week
# day_of_mt = timeline.day
# day_of_yr = timeline.day_of_year

dow_cos = np.cos(2 * np.pi /   7 * day_of_wk)
dow_sin = np.sin(2 * np.pi /   7 * day_of_wk)
# dom_cos = np.cos(2 * np.pi /  31 * day_of_mt)
# dom_sin = np.sin(2 * np.pi /  31 * day_of_mt)
# doy_cos = np.cos(2 * np.pi / 365 * day_of_yr)
# doy_sin = np.sin(2 * np.pi / 365 * day_of_yr)


In [ ]:
def fit_model() -> np.ndarray:
    ...
    # add cyclif FE - seasons, day of year sin and cos

In [ ]:
eps = r_eps[-1]
eps.std(axis=0)

np.float64(0.00448524182445094)

In [ ]:
# Two forms of regression happning
# 1. Factor removal from returns - residual returns
# 2. Fittign the OLS model - Alpha model

# NOTE start by precomputing the residual returns at each time step
# targets and features in blocks of data
# preprocess all data and save down to npz,
# then run model fit loop on top of that




# %%
# NOTE Regress Sectors and International ETFs on market (potentially gold and bonds as well)
# Rank idiosyncratic returns to get assets that perform well orthogonally to the market
# Otherwise predictor may rank which ETF has biggest market component

# XLK - Technology
# XLV - Heathcare
# XLF - Financials
# XLY - Consumer Discretionary
# XLI - Industrials
# XLP - Consumer Staples
# XLE - Energy
# XLU - Utilities
# XLB - Materials

# IBB - Biotech
# IYR - Real Estate

# rd[[
#     "SPY", "AGG", "GLD",
#     "XLK", "XLV", "XLF", "XLY", "XLI", "XLP", "XLE", "XLU", "XLB",  # sectors
#     "IBB", "IYR",  # extra
#     # "EWJ", "EWG", "EWU", "EWA", "EWH", "EWS", "EWZ", "EWT", "EWY", "EWP", "EWW", "EWD", "EWL", "EWC",  # international
# ]].corr()


# r = B * rm + e
# e = r - B * rm
# rm = rx[["SPY"]].values
# r  = rx[sect].values
# Beta = r.T @ rm / (rm.T @ rm)
# eps = r - Beta.T * rm
# df_beta = pd.Series(Beta.ravel(), index=sect)
# df_eps = pd.DataFrame(eps, columns=sect)
# df_eps.corr()